# Task 2: Clean pipeline

Build `cleaned_parcel_timeseries.csv` by applying every cleanup decision from [Readme.md](Readme.md).

**Steps:** setup → helpers → load → clean readings → clean metadata → join → validate → write → preview

In [1]:
from pathlib import Path
import re
from datetime import date

import polars as pl
from dateutil import parser as date_parser

# pip install python-dateutil  (if not already installed)

ROOT = Path(".")
READINGS_PATH = ROOT / "resources" / "parcel_readings.csv"
META_PATH = ROOT / "resources" / "parcel_metadata.csv"
OUTPUT_PATH = ROOT / "cleaned_parcel_timeseries.csv"

NDVI_MIN = -1.0
NDVI_MAX = 1.0

READINGS_COLS = [
    "parcel_id",
    "date",
    "ndvi_value",
    "temperature_c",
    "rainfall_mm",
    "sensor_status",
]

META_COLS = [
    "parcel_id",
    "mill_id",
    "crop_type",
    "sowing_date",
    "area_hectares",
]

FINAL_COLS = [
    "parcel_id",
    "date",
    "ndvi_value",
    "temperature_c",
    "rainfall_mm",
    "sensor_status",
    "mill_id",
    "crop_type",
    "sowing_date",
    "area_hectares",
    "is_data_ambiguous",
    "is_ndvi_out_of_range",
    "is_duplicate",
    "is_parcel_id_missing",
    "is_date_missing",
    "is_ndvi_missing",
    "is_temperature_missing",
    "is_rainfall_missing",
    "is_sensor_status_missing",
]

In [2]:
SLASH_DATE_RE = re.compile(r"^(\d{2})/(\d{2})/(\d{4})$")
ISO_DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")


def is_ambiguous_slash(date_raw: str) -> bool:
    m = SLASH_DATE_RE.match(date_raw.strip())
    if not m:
        return False
    p0, p1 = int(m.group(1)), int(m.group(2))
    return p0 <= 12 and p1 <= 12


def infer_slash_use_mdy(date_raws: list[str]) -> bool:
    """Vote MDY vs DMY from unambiguous slash dates; default DMY on tie."""
    evidence_dmy = evidence_mdy = 0
    for s in date_raws:
        m = SLASH_DATE_RE.match(s.strip())
        if not m:
            continue
        p0, p1 = int(m.group(1)), int(m.group(2))
        if p0 > 12:
            evidence_dmy += 1
        elif p1 > 12:
            evidence_mdy += 1
    return evidence_mdy > evidence_dmy


def parse_reading_date(date_raw: str, use_mdy: bool) -> tuple[date | None, int]:
    s = (date_raw or "").strip()
    if not s:
        return None, 0
    ambiguous = is_ambiguous_slash(s)
    try:
        if ISO_DATE_RE.match(s):
            parsed = date_parser.parse(s, dayfirst=False)
        elif ambiguous or SLASH_DATE_RE.match(s):
            parsed = date_parser.parse(s, dayfirst=not use_mdy)
        else:
            parsed = date_parser.parse(s, dayfirst=True)
        return parsed.date(), int(ambiguous)
    except (ValueError, TypeError, OverflowError):
        return None, int(ambiguous)


def empty_or_null_mask(col: str) -> pl.Expr:
    stripped = pl.col(col).str.strip_chars()
    return stripped.is_null() | (stripped == "") | stripped.str.to_uppercase().is_in(
        ["NA", "NAN", "NULL", "NONE"]
    )


def normalize_sensor_status(col: str = "sensor_status") -> pl.Expr:
    stripped = pl.col(col).str.strip_chars().str.to_uppercase()
    return (
        pl.when(pl.col(col).is_null() | stripped.is_null())
        .then(pl.lit("MISSING"))
        .when(stripped.is_in(["", "NA", "NAN", "NULL", "NONE"]))
        .then(pl.lit("MISSING"))
        .when(stripped == "OK")
        .then(pl.lit("OK"))
        .when(stripped == "ERROR")
        .then(pl.lit("ERROR"))
        .otherwise(stripped)
    )

### Load data

In [3]:
readings = pl.read_csv(
    READINGS_PATH,
    schema_overrides={col: pl.Utf8 for col in READINGS_COLS},
)
metadata = pl.read_csv(
    META_PATH,
    schema_overrides={col: pl.Utf8 for col in META_COLS},
)

print(f"Readings: {readings.height:,} rows")
print(f"Metadata: {metadata.height:,} rows")

Readings: 3,447 rows
Metadata: 28 rows


### Clean readings

In [12]:
def clean_readings(df: pl.DataFrame) -> pl.DataFrame:
    df = (
        df.with_row_index("_row_idx")
        .with_columns(
            pl.col("parcel_id").str.strip_chars(),
            pl.col("date").str.strip_chars().alias("date_raw"),
        )
        .drop("date")  # avoid duplicate 'date' when unnesting _parsed
    )

    df = df.with_columns(
        empty_or_null_mask("parcel_id").cast(pl.Int8).alias("is_parcel_id_missing"),
        empty_or_null_mask("date_raw").cast(pl.Int8).alias("is_date_missing"),
    )

    use_mdy = infer_slash_use_mdy(df["date_raw"].to_list())
    print(f"We are using MDY for parsing ambiguous dates:{use_mdy}")

    def _parse_row(s: str) -> dict:
        parsed_date, is_ambiguous = parse_reading_date(s, use_mdy)
        return {"date": parsed_date, "is_data_ambiguous": is_ambiguous}

    df = df.with_columns(
        pl.col("date_raw")
        .map_elements(
            _parse_row,
            return_dtype=pl.Struct(
                {"date": pl.Date, "is_data_ambiguous": pl.Int8}
            ),
        )
        .alias("_parsed")
    ).unnest("_parsed")

    df = df.with_columns(
        (
            (pl.col("is_date_missing") == 1) | pl.col("date").is_null()
        )
        .cast(pl.Int8)
        .alias("is_date_missing")
    )

    # Required keys: drop rows with missing parcel_id or date
    df = df.filter(
        (pl.col("is_parcel_id_missing") == 0) & (pl.col("is_date_missing") == 0)
    )

    ndvi_raw_missing = empty_or_null_mask("ndvi_value")
    temp_raw_missing = empty_or_null_mask("temperature_c")
    rain_raw_missing = empty_or_null_mask("rainfall_mm")
    status_raw_missing = empty_or_null_mask("sensor_status")

    ndvi_f = pl.col("ndvi_value").cast(pl.Float64, strict=False)
    ndvi_out = (ndvi_f < NDVI_MIN) | (ndvi_f > NDVI_MAX)
    temp_f = pl.col("temperature_c").cast(pl.Float64, strict=False)
    rain_f = pl.col("rainfall_mm").cast(pl.Float64, strict=False)
    sensor_status = normalize_sensor_status()

    df = df.with_columns(
        pl.when(ndvi_out).then(None).otherwise(ndvi_f).alias("ndvi_value"),
        ndvi_out.cast(pl.Int8).alias("is_ndvi_out_of_range"),
        pl.when(temp_raw_missing).then(None).otherwise(temp_f).alias("temperature_c"),
        pl.when(rain_raw_missing).then(None).otherwise(rain_f).alias("rainfall_mm"),
        sensor_status.alias("sensor_status"),
        (
            ndvi_raw_missing | (ndvi_f.is_null() & ~ndvi_out)
        )
        .cast(pl.Int8)
        .alias("is_ndvi_missing"),
        (
            temp_raw_missing | (temp_f.is_null() & ~temp_raw_missing)
        )
        .cast(pl.Int8)
        .alias("is_temperature_missing"),
        (
            rain_raw_missing | (rain_f.is_null() & ~rain_raw_missing)
        )
        .cast(pl.Int8)
        .alias("is_rainfall_missing"),
        ((sensor_status == "MISSING") | status_raw_missing)
        .cast(pl.Int8)
        .alias("is_sensor_status_missing"),
    )

    df = df.with_columns(
        (
            pl.col("ndvi_value").is_not_null()
            & (pl.col("ndvi_value") >= NDVI_MIN)
            & (pl.col("ndvi_value") <= NDVI_MAX)
        ).alias("_ndvi_ok"),
        (pl.col("sensor_status") == "OK").alias("_status_ok"),
        pl.len().over(["parcel_id", "date"]).alias("_dup_n"),
    )

    df = (
        df.sort(["_ndvi_ok", "_status_ok", "_row_idx"], descending=[True, True, False])
        .group_by(["parcel_id", "date"], maintain_order=True)
        .first()
    )

    return df.with_columns(
        (pl.col("_dup_n") > 1).cast(pl.Int8).alias("is_duplicate"),
    ).drop("date_raw", "_row_idx", "_ndvi_ok", "_status_ok", "_dup_n")

### Clean metadata

In [13]:
def clean_metadata(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        pl.col("parcel_id").str.strip_chars(),
        pl.col("mill_id").str.strip_chars(),
        pl.col("crop_type").str.strip_chars(),
        pl.col("sowing_date").str.to_date("%Y-%m-%d", strict=False),
        pl.col("area_hectares").cast(pl.Float64, strict=False),
    )

### Run pipeline

In [14]:
readings_clean = clean_readings(readings)
metadata_clean = clean_metadata(metadata)

print(f"Readings after clean (pre-join): {readings_clean.height:,} rows")

timeseries = readings_clean.join(metadata_clean, on="parcel_id", how="inner").select(
    FINAL_COLS
)

print(f"Timeseries after inner join: {timeseries.height:,} rows")

We are using MDY for parsing ambiguous dates:False
Readings after clean (pre-join): 3,439 rows
Timeseries after inner join: 3,399 rows


### Validate

In [7]:
assert timeseries.filter(pl.col("parcel_id").is_null() | pl.col("date").is_null()).is_empty()
assert timeseries.filter(
    (pl.col("is_parcel_id_missing") == 1) | (pl.col("is_date_missing") == 1)
).is_empty()

dup_check = (
    timeseries.group_by("parcel_id", "date")
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
)
assert dup_check.is_empty(), f"Duplicate keys remain: {dup_check.height}"

dedup_removed = readings.height - readings_clean.height
orphan_excluded = readings_clean.height - timeseries.height

meta_no_readings = (
    metadata_clean.select("parcel_id")
    .join(readings.select("parcel_id").unique(), on="parcel_id", how="anti")
    .sort("parcel_id")
)

print("Validation summary")
print(f"  Raw readings: {readings.height:,}")
print(f"  After clean (dedupe): {readings_clean.height:,} (removed {dedup_removed} duplicate rows)")
print(f"  After inner join: {timeseries.height:,} (excluded {orphan_excluded} orphan rows)")
print(f"  is_data_ambiguous=1: {timeseries['is_data_ambiguous'].sum()}")
print(f"  is_ndvi_out_of_range=1: {timeseries['is_ndvi_out_of_range'].sum()}")
print(f"  is_duplicate=1: {timeseries['is_duplicate'].sum()}")
print(f"  is_ndvi_missing=1: {timeseries['is_ndvi_missing'].sum()}")
print(f"  is_temperature_missing=1: {timeseries['is_temperature_missing'].sum()}")
print(f"  is_rainfall_missing=1: {timeseries['is_rainfall_missing'].sum()}")
print(f"  is_sensor_status_missing=1: {timeseries['is_sensor_status_missing'].sum()}")
print("  sensor_status counts:")
print(timeseries.group_by("sensor_status").len().sort("len", descending=True))
print(
    f"  Date range: {timeseries['date'].min()} to {timeseries['date'].max()}"
)
print(f"  Metadata parcels with no readings ({meta_no_readings.height}):")
print(meta_no_readings)

Validation summary
  Raw readings: 3,447
  After clean (dedupe): 3,439 (removed 8 duplicate rows)
  After inner join: 3,399 (excluded 40 orphan rows)
  is_data_ambiguous=1: 275
  is_ndvi_out_of_range=1: 104
  is_duplicate=1: 8
  is_ndvi_missing=1: 0
  is_temperature_missing=1: 0
  is_rainfall_missing=1: 0
  is_sensor_status_missing=1: 137
  sensor_status counts:
shape: (3, 2)
┌───────────────┬──────┐
│ sensor_status ┆ len  │
│ ---           ┆ ---  │
│ str           ┆ u32  │
╞═══════════════╪══════╡
│ OK            ┆ 3017 │
│ ERROR         ┆ 245  │
│ MISSING       ┆ 137  │
└───────────────┴──────┘
  Date range: 2026-01-01 to 2026-05-31
  Metadata parcels with no readings (3):
shape: (3, 1)
┌────────────┐
│ parcel_id  │
│ ---        │
│ str        │
╞════════════╡
│ PARCEL_050 │
│ PARCEL_051 │
│ PARCEL_052 │
└────────────┘


### Write output

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
# quote_style=always: null ndvi writes as "" so Excel keeps columns aligned (not ,,)
timeseries.write_csv(OUTPUT_PATH, null_value="", quote_style="always")
print(f"Wrote {timeseries.height:,} rows to {OUTPUT_PATH}")

Wrote 3,399 rows to cleaned_parcel_timeseries.csv


### Preview

In [9]:
display(timeseries.head(10))
print("\nRows per crop_type:")
display(timeseries.group_by("crop_type").len().sort("len", descending=True))

parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status,mill_id,crop_type,sowing_date,area_hectares,is_data_ambiguous,is_ndvi_out_of_range,is_duplicate,is_parcel_id_missing,is_date_missing,is_ndvi_missing,is_temperature_missing,is_rainfall_missing,is_sensor_status_missing
str,date,f64,f64,f64,str,str,str,date,f64,i8,i8,i8,i8,i8,i8,i8,i8,i8
"""PARCEL_014""",2026-01-27,0.457,27.6,0.0,"""OK""","""MILL_NORTH""","""sugarcane""",2026-01-05,9.39,0,0,0,0,0,0,0,0,0
"""PARCEL_006""",2026-05-17,0.497,25.8,0.0,"""OK""","""MILL_WEST""","""sugarcane""",2026-02-11,5.67,0,0,0,0,0,0,0,0,0
"""PARCEL_004""",2026-02-10,0.81,25.0,0.0,"""OK""","""MILL_NORTH""","""sugarcane""",2026-01-02,3.18,1,0,0,0,0,0,0,0,0
"""PARCEL_002""",2026-01-17,0.269,33.2,0.0,"""OK""","""MILL_SOUTH""","""sugarcane""",2026-01-16,8.97,0,0,0,0,0,0,0,0,0
"""PARCEL_015""",2026-03-19,0.873,26.3,0.0,"""OK""","""MILL_SOUTH""","""sugarcane""",2026-01-06,4.87,0,0,0,0,0,0,0,0,0
"""PARCEL_020""",2026-05-08,0.806,16.8,2.7,"""OK""","""MILL_NORTH""","""sugarcane""",2026-02-19,9.95,0,0,0,0,0,0,0,0,0
"""PARCEL_010""",2026-01-20,0.315,27.9,0.9,"""OK""","""MILL_EAST""","""sugarcane""",2026-01-07,7.44,0,0,0,0,0,0,0,0,0
"""PARCEL_019""",2026-01-10,0.198,15.3,1.0,"""OK""","""MILL_SOUTH""","""sugarcane""",2026-01-18,10.19,0,0,0,0,0,0,0,0,0
"""PARCEL_010""",2026-05-10,0.403,18.7,0.0,"""OK""","""MILL_EAST""","""sugarcane""",2026-01-07,7.44,1,0,0,0,0,0,0,0,0



Rows per crop_type:


crop_type,len
str,u32
"""sugarcane""",2587
"""soybean""",544
"""wheat""",268
